# Three checks: de-contaminate, test the premise, sweep the layers

Phase 0.1 + P.1 + 1.1 of `VISUAL_TOKEN_STRATEGY.md`. Reuses the cached
`wearvqa_faithfulness.pt` ground truth and the frozen `sink_mask_smolvlm2.pt`, so the expensive
81-ablation pass is **not** repeated.

| check | question | decides | cost |
|---|---|---|---|
| **1** | precision@k with sinks removed from **both** the ground truth and the labels | whether `imp_a raw`'s 38% was just sink agreement, and whether `imp_context` leads once both sides are corrected | free |
| **2** | how far is the answer's most important patch from the gaze? | whether WEAR-VQA contains the far-context phenomenon FRM exists for — **can retire the dataset** | free |
| **3** | layer band sweep: all / shallow / middle / deep | whether "attention is uninformative" was an aggregation bug | ~2 min GPU |

Every comparison so far had the sink on one side only: the labels corrected for it, the LOO ground
truth did not. They therefore disagreed at the single largest value in the vector. Check 1 fixes that.

> Order matters. Check 2 is only meaningful on a de-contaminated ground truth. Check 3 tells you
> about attention quality either way, but is only worth *acting* on if Check 2 says the data has signal.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip -q install -U "transformers>=4.49" accelerate huggingface_hub safetensors pillow num2words matplotlib scipy
!rm -rf /content/text_vision_attention_map
!git clone -q https://github.com/shubhamOjha1000/text_vision_attention_map.git /content/text_vision_attention_map
%cd /content/text_vision_attention_map

In [ ]:
import importlib.util, os, sys, math
import numpy as np
import torch
import matplotlib.pyplot as plt
from collections import defaultdict

sys.path.insert(0, os.getcwd())

def _load(mod, rel):
    spec = importlib.util.spec_from_file_location(mod, os.path.join(os.getcwd(), rel))
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m

S = _load("probe_smolvlm", "tests/probe_smolvlm.py")
import rater_selection as RS
import visual_selection as VS

CACHE = "/content/drive/MyDrive/wearvqa_faithfulness.pt"
SINKF = "/content/drive/MyDrive/sink_mask_smolvlm2.pt"
KS    = (5, 10, 12)

assert os.path.exists(CACHE), f"{CACHE} missing - run colab_faithfulness_loo.ipynb"
data = torch.load(CACHE, weights_only=False)
L_v  = data[0]["drops"].numel()
G    = int(round(math.sqrt(L_v)))
N    = len(data)

if os.path.exists(SINKF):
    sinks = torch.load(SINKF, weights_only=False)["sink_mask"].bool()
    print(f"loaded frozen sink mask: {int(sinks.sum())} tokens "
          f"{torch.nonzero(sinks).squeeze(-1).tolist()}")
else:                                    # fall back: re-derive from the cached importances
    print("sink mask not on Drive - deriving from the question-invariant baseline")
    sinks = VS.sink_token_mask(VS.make_baseline([d["imp_q"] for d in data]), 7)

cand      = VS.candidate_mask(L_v, exclude=[sinks])
cand_idx  = torch.nonzero(cand, as_tuple=False).squeeze(-1)
n_cand    = int(cand.sum())

# PMI labels (the cache stores only raw importances)
bq = VS.make_baseline_loo([d["imp_q"] for d in data])
ba = VS.make_baseline_loo([d["imp_a"] for d in data])
for i, d in enumerate(data):
    d["pmi_q"]   = VS.pmi_scores(d["imp_q"], bq[i])
    d["pmi_a"]   = VS.pmi_scores(d["imp_a"], ba[i])
    d["context"] = torch.relu(d["pmi_a"] - d["pmi_q"])

print(f"\n{N} examples | L_v={L_v} ({G}x{G}) | candidates after sink removal: {n_cand}")
print("chance precision@k over candidates: "
      + ", ".join(f"k={k}: {k / n_cand:.1%}" for k in KS))

## Check 1 — precision@k with sinks removed from BOTH sides

First a usability diagnostic. The sink was the top-drop patch in 15/20 examples, so removing it may
leave the ground truth with nothing but near-zero drops. **If most examples become unusable, the LOO
signal was almost entirely the sink** — which is itself the answer.

In [ ]:
nz = [int((d["drops"][cand_idx].abs() > 1e-6).sum()) for d in data]
print("nonzero drops among the 74 candidates, per example:")
print("  ", sorted(nz, reverse=True))
print(f"   median {np.median(nz):.0f}   examples with < 10 nonzero: {sum(x < 10 for x in nz)}/{N}")

drop_share = [float(d["drops"][sinks].abs().sum() / (d["drops"].abs().sum() + 1e-12)) for d in data]
print(f"\nshare of total |drop| held by the {int(sinks.sum())} sink tokens: "
      f"mean {np.mean(drop_share):.1%}  median {np.median(drop_share):.1%}")
print("   -> how much of the 'ground truth' was actually sink-removal damage")

In [ ]:
def topk_cand(v, k):
    """top-k restricted to the candidate set, returned as ORIGINAL token indices."""
    return set(cand_idx[torch.topk(v[cand_idx], k).indices].tolist())

g = torch.Generator().manual_seed(0)

def label_set(d):
    gp = min(G - 1, int(d["gaze"]["y_norm"] * G)) * G + min(G - 1, int(d["gaze"]["x_norm"] * G))
    def distmap(c):
        r0, c0 = divmod(c, G)
        return torch.tensor([-math.hypot(i // G - r0, i % G - c0) for i in range(L_v)])
    return {
        "imp_q  raw":         d["imp_q"],
        "imp_q  pmi":         d["pmi_q"],
        "imp_a  raw":         d["imp_a"],
        "imp_a  pmi":         d["pmi_a"],
        "imp_context (a-q)":  d["context"],
        "CTRL reverse (q-a)": torch.relu(d["pmi_q"] - d["pmi_a"]),
        "gaze proximity":     distmap(gp),
        "center (no image)":  distmap(L_v // 2),
        "CTRL random":        torch.rand(L_v, generator=g),
    }

# numbers from the previous, sink-CONTAMINATED run, for side-by-side comparison
PREV = {5:  {"imp_a  raw": .380, "imp_q  raw": .290, "imp_a  pmi": .270, "imp_q  pmi": .190,
             "imp_context (a-q)": .180, "gaze proximity": .120, "CTRL random": .090,
             "center (no image)": .080, "CTRL reverse (q-a)": .050},
        10: {"imp_a  raw": .384, "imp_a  pmi": .311, "imp_q  raw": .295, "imp_context (a-q)": .226,
             "imp_q  pmi": .200, "gaze proximity": .189, "CTRL random": .153,
             "center (no image)": .100, "CTRL reverse (q-a)": .084},
        12: {"imp_a  raw": .361, "imp_a  pmi": .319, "imp_q  raw": .306, "imp_q  pmi": .245,
             "imp_context (a-q)": .241, "gaze proximity": .213, "CTRL random": .153,
             "center (no image)": .130, "CTRL reverse (q-a)": .106}}

from scipy.stats import norm
for k in KS:
    chance = k / n_cand
    rows, used = defaultdict(list), 0
    for d in data:
        if float(torch.topk(d["drops"][cand_idx], k).values[-1]) <= 1e-6:
            continue                      # ground-truth top-k not well defined
        used += 1
        gt = topk_cand(d["drops"], k)
        for name, v in label_set(d).items():
            rows[name].append(len(topk_cand(v, k) & gt) / k)

    print(f"\n=== precision@{k}   chance {chance:.1%}   usable {used}/{N} examples")
    if used < 4:
        print("   too few usable examples - the ground truth is empty once sinks are removed")
        continue
    print(f"{'label':<22}{'corrected':>11}{'was':>8}{'delta':>8}{'p':>8}")
    print("-" * 57)
    for name in sorted(rows, key=lambda n: -np.mean(rows[n])):
        a = np.array(rows[name])
        se = a.std(ddof=1) / max(np.sqrt(len(a)), 1e-9)
        p = 2 * (1 - norm.cdf(abs((a.mean() - chance) / se))) if se > 0 else 1.0
        was = PREV[k].get(name, float("nan"))
        print(f"{name:<22}{a.mean():>11.1%}{was:>8.1%}{a.mean() - was:>+8.1%}"
              f"{p:>8.3f}{'  *' if p < 0.05 else ''}")

### How to read Check 1

* `imp_a raw` **drops sharply** -> its 38% was sink agreement, as the clean-5 subset suggested.
* `imp_context` **holds or rises** -> it was the only label that improved when sinks were
  stratified out; this confirms it on all 20 examples instead of 5.
* **Everything collapses to chance** -> the LOO ground truth carried no content signal beyond the
  sink. Go to Phase 0.2/0.3 (grouped ablation, deletion curves) before drawing any conclusion about
  the labels.

## Check 2 — is there any far context in this dataset?

For each example, the distance from the **gaze patch** to the **highest-drop patch among candidates**
— i.e. how far from the eye the information the answer actually needed was sitting.

Three outcomes, three very different consequences:

* **distance ~ 0** -> the answer needed what you were already looking at. Stage 2a's fovea covers it,
  and FRM has nothing left to retrieve.
* **distance ~ chance** -> gaze carries no information about where the important patch is, so FRM
  cannot learn the mapping either.
* **concentrated at a non-zero offset** -> gaze predicts a displaced region. That is the FRM premise.

The offset scatter matters more than the mean distance: FRM learns `E_q[imp | gaze]`, so what it
needs is for the offset *distribution* to be non-uniform, not merely large.

In [ ]:
def gaze_patch(gz):
    return min(G - 1, int(gz["y_norm"] * G)) * G + min(G - 1, int(gz["x_norm"] * G))

rows, offs = [], []
for d in data:
    if float(d["drops"][cand_idx].max()) <= 1e-6:
        continue                                  # no meaningful ground truth
    gp   = gaze_patch(d["gaze"])
    top  = int(cand_idx[d["drops"][cand_idx].argmax()])
    gr, gc = divmod(gp, G); tr, tc = divmod(top, G)
    rows.append(dict(type=d["type"], dist=math.hypot(tr - gr, tc - gc),
                     chance=float(np.mean([math.hypot(i // G - gr, i % G - gc)
                                           for i in cand_idx.tolist()]))))
    offs.append((tc - gc, tr - gr))

dist   = np.array([r["dist"] for r in rows])
chance = np.array([r["chance"] for r in rows])
print(f"usable examples: {len(rows)}/{N}\n")
print(f"gaze -> top-drop patch distance (grid cells)")
print(f"   median   {np.median(dist):.2f}")
print(f"   mean     {dist.mean():.2f}   vs chance {chance.mean():.2f}")
print(f"   within 1 cell of gaze : {(dist <= 1.5).sum()}/{len(dist)}  <- fovea already covers these")
print(f"   beyond 3 cells        : {(dist > 3).sum()}/{len(dist)}  <- the FAR-CONTEXT cases")

by = defaultdict(list)
for r in rows:
    by[r["type"]].append(r["dist"])
print("\nper question type (mean distance):")
for t in sorted(by):
    v = np.mean(by[t])
    print(f"   {t:<38} {v:4.2f}{'   <- far context' if v > 3 else ''}")

fig, ax = plt.subplots(1, 2, figsize=(10.5, 4.2))
ax[0].hist(dist, bins=np.arange(0, G + 1, 0.75), color="steelblue")
ax[0].axvline(chance.mean(), color="r", ls="--", label=f"chance {chance.mean():.1f}")
ax[0].axvline(1.5, color="g", ls=":", label="fovea reach")
ax[0].set_xlabel("gaze -> top-drop distance (cells)"); ax[0].set_ylabel("examples")
ax[0].set_title("How far was the information?"); ax[0].legend(fontsize=8)

dx, dy = zip(*offs)
ax[1].scatter(dx, dy, s=45, alpha=.75)
ax[1].scatter([0], [0], marker="x", s=160, c="lime", linewidths=3)
ax[1].axhline(0, lw=.5, c="k"); ax[1].axvline(0, lw=.5, c="k")
ax[1].set_xlim(-G, G); ax[1].set_ylim(G, -G)
ax[1].set_xlabel("column offset from gaze"); ax[1].set_ylabel("row offset from gaze")
ax[1].set_title("Offset distribution (clustered = learnable)")
plt.tight_layout(); plt.show()

print(f"\noffset spread: col std {np.std(dx):.2f}, row std {np.std(dy):.2f}  "
      f"| mean offset ({np.mean(dx):+.2f}, {np.mean(dy):+.2f})")
print("a tight cluster away from the origin = FRM has something to learn;")
print("a cloud filling the square = gaze does not predict where the context is.")

## Check 3 — layer band sweep (GPU)

`default_band` averages **all** decoder layers. LearnPruner reports text->vision attention is
reliable in **middle** layers and degrades in deep ones; a separate study finds random pruning
surpasses every method by layer 14. So averaging everything mixes the reliable with the actively
harmful.

Re-extracts attention for the 20 examples (~2 min), then recomputes importance per band with sinks
excluded **before** the softmax.

In [ ]:
model, processor, device = S._load_smolvlm("HuggingFaceTB/SmolVLM2-2.2B-Instruct")
tokenizer = processor.tokenizer

def build_inputs(image, question, answer):
    msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": question}]}]
    prompt = processor.apply_chat_template(msgs, add_generation_prompt=True)
    full = processor(text=prompt + answer, images=[image], return_tensors="pt").to(device)
    only = processor(text=prompt, images=[image], return_tensors="pt")
    return full, int(only["input_ids"].shape[1])

def raw_scores_for(inp):
    patched = S._patch_eager_globals(S._make_raw_capturing_eager(None))
    try:
        with torch.no_grad():
            model(**inp)
    finally:
        S._unpatch_eager_globals(patched)
    L, raw = int(inp["input_ids"].shape[1]), {}
    for m in model.modules():
        r = getattr(m, "_raw_attn_scores", None)
        if r is not None and r.shape[-1] == L and r.shape[-2] == L:
            raw[int(getattr(m, "layer_idx", len(raw)))] = r[0].float()
        for a in ("_raw_attn_scores", "_post_attn"):
            if hasattr(m, a):
                delattr(m, a)
    return raw

per_ex = []
for i, d in enumerate(data):
    img = S.load_image(d["img_path"])
    inp, n_prompt = build_inputs(img, d["question"], d["answer"])
    ids = inp["input_ids"][0].cpu()
    iid = S._find_image_token_id(model, processor)
    pad = tokenizer.pad_token_id
    im  = ids == iid
    tm  = (ids != iid) & (ids != (pad if pad is not None else -10**9))
    maps, tpos, _ = RS.sliced_maps_from_full(raw_scores_for(inp), im, tm)
    tt  = tokenizer.convert_ids_to_tokens(ids[tpos].tolist())
    rq  = RS.select_important_text_tokens(maps, text_tokens=tt, tokenizer=tokenizer,
                                          question=d["question"], pct=0.5).rater_mask
    isa = torch.tensor([int(p) >= n_prompt for p in tpos.tolist()])
    ra  = RS.content_text_mask(tt, tokenizer) & isa
    if int(ra.sum()) == 0:
        ra = isa
    per_ex.append((maps, rq, ra))
    if (i + 1) % 5 == 0:
        print(f"  {i + 1}/{N}")

n_layers = len(per_ex[0][0])
t = max(1, n_layers // 3)
BANDS = {"all":     list(range(n_layers)),
         "shallow": list(range(0, t)),
         "middle":  list(range(t, 2 * t)),
         "deep":    list(range(2 * t, n_layers)),
         f"single L{n_layers // 2}": [n_layers // 2]}
print(f"\n{n_layers} decoder layers -> bands "
      + ", ".join(f"{k}:{v[0]}-{v[-1]}" for k, v in BANDS.items()))

In [ ]:
K = 10
chance = K / n_cand
usable = [i for i, d in enumerate(data)
          if float(torch.topk(d["drops"][cand_idx], K).values[-1]) > 1e-6]
print(f"precision@{K} by layer band   (chance {chance:.1%}, {len(usable)}/{N} usable)\n")
print(f"{'band':<12}{'rows':<8}{'prec@10':>9}{'entropy %':>11}{'peak/unif':>11}")
print("-" * 51)

best = {}
for bname, band in BANDS.items():
    for rows_name, ridx in (("imp_q", 1), ("imp_a", 2)):
        precs, ents, peaks = [], [], []
        for i in usable:
            maps, rq, ra = per_ex[i]
            mask = rq if ridx == 1 else ra
            imp, *_ = VS.image_importance(maps, mask, band=band, cand_mask=cand)
            gt = topk_cand(data[i]["drops"], K)
            precs.append(len(topk_cand(imp, K) & gt) / K)
            p = imp[cand_idx]
            ents.append(float(-(p[p > 0] * p[p > 0].log()).sum()) / math.log(n_cand))
            peaks.append(float(p.max()) * n_cand)
        m = np.mean(precs)
        best[(bname, rows_name)] = m
        print(f"{bname:<12}{rows_name:<8}{m:>9.1%}{np.mean(ents):>10.1%}{np.mean(peaks):>11.1f}x")

top = max(best, key=best.get)
print(f"\nbest: {top[0]} band, {top[1]} rows -> {best[top]:.1%} "
      f"({best[top] / chance:.2f}x chance)")
print(f"vs 'all layers' baseline: imp_a {best[('all','imp_a')]:.1%}, "
      f"imp_q {best[('all','imp_q')]:.1%}")

## Verdict

Fill in from the printed tables, not from intuition.

**Check 1**
* `imp_a raw` falls a lot -> its lead was sink agreement (expected).
* `imp_context` holds/rises and beats `CTRL random` -> it is the label to carry forward.
* Everything at chance, or too few usable examples -> the LOO ground truth was mostly sink.
  Do Phase 0.2 (`GROUP=2`) and 0.3 (deletion curves) before judging any label.

**Check 2**
* median distance <= 1.5 and a tight cluster at the origin -> **the dataset has no far context.**
  Stage 2a already covers what the answer needed. Change dataset (Ego-Exo4D / EGTEA) before
  spending anything more on the teacher.
* mean ~ chance and a cloud filling the square -> gaze does not predict where the context is;
  FRM cannot learn what is not there.
* a tight cluster at a non-zero offset -> the premise holds. Proceed.

**Check 3**
* middle >> all -> the "attention is uninformative" conclusion was an aggregation artifact. Change
  `default_band` and re-run every earlier experiment.
* all bands equal and near chance -> layer choice is not the problem; move to Phase 2 (test-time
  registers) and Phase 3 (rollout / attention x gradient / diversity teachers).
* entropy still ~95% in every band -> the distribution has no concentration to find, and the
  affordable-LOO shortcut (~35 min for 2500 examples at `GROUP=2`) becomes the main path.